In [1]:
import torch
import torch.nn as nn

In [2]:
block_size = 8 # n-1 gram
embedding_size = 16
head_size = 512
batch_size = 512
steps = 20000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=block_size, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=block_size, step=1)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [4]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size),
            nn.Flatten(),
            nn.Linear(in_features=embedding_size * block_size, out_features=head_size),
            nn.ReLU(),
            nn.Linear(in_features=head_size, out_features=vocab_size)
        )

    def forward(self, input):
        return self.head(input=input)

In [5]:
# model initialization
model = MultiLayerPerceptron()
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    train_logits = model(input=x_train[batch])
    train_loss = nn.functional.cross_entropy(input=train_logits, target=y_train[batch])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits, target=y_val)
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.221597194671631, val loss = 4.227893829345703


Step 1000: train loss = 1.7342506647109985, val loss = 1.9042882919311523


Step 2000: train loss = 1.5628647804260254, val loss = 1.832755208015442


Step 3000: train loss = 1.6222580671310425, val loss = 1.803411841392517


Step 4000: train loss = 1.5579822063446045, val loss = 1.7846789360046387


Step 5000: train loss = 1.4727065563201904, val loss = 1.7784510850906372


... (11 outputs left out)

Step 17000: train loss = 1.4798610210418701, val loss = 1.725663423538208


Step 18000: train loss = 1.365256905555725, val loss = 1.7127102613449097


Step 19000: train loss = 1.4336414337158203, val loss = 1.7099456787109375


Step 19999: train loss = 1.4097182750701904, val loss = 1.708356499671936


In [7]:
# prediction
with torch.inference_mode():
    output = ['\n'] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)
        
        logits = model(input=i)
        P = torch.softmax(input=logits, dim=1)

        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [8]:
print(''.join(output[block_size:]))

NEAMIOR:
Swellow bring suk?

CORIOLANUS:
No, iffection. Sir, since for to slanded and full city,
That madeers ere great whicherly.

BRUTUS:
Have likely; hore it,
Tiscouse to thim?

Secannot my grow!
Herp inound the name insuided, concing.

BRUTUS:
Nay, madam, who if happy:
Iitlan's master.

CORIOLANUS:
What is't take
know her conques that passable save hour lateful a point;
Forth, foul caps is gentlemen hear me, in here?

SIR Eday.

BRUTUS:
Look'd hen. all the Volson, now, go, general.

ROMEO:
I grace shown, grave with our love tilet your soly prove me,
Unchanm'd.
The bear yours.

KING RICHARD:
I am it word,
Where in you:
Trudge, thee rost of colity,
And gands would am: madam, and save at fal the ragon at take. Camillor is will.

Second Mercy, on your cousin, me forsworn since? Be converce so which on them; thou hash your to a go tin,
The veler, a this?
Well, to will skille!
Thou, benefil serve unctless of Marcitizans: wife arm mack,
Whilver should slain, man.  He back, sir.

B
